## 04. scratchpad

### 1. Data Upload

In [54]:
import pandas as pd
import os
from collections import defaultdict, Counter
import re
import time

In [2]:
def upload_to_dataframe(root, files, num_lines):
    path_eng, path_pol = [root+f for f in files]
    data = defaultdict(list)
    
    with open (path_eng, encoding="utf_8") as f_eng, open (path_pol, encoding="utf_8") as f_pol:
        for _ in range(num_lines):
            data['eng_text'].append(f_eng.readline().strip())
            data['pol_text'].append(f_pol.readline().strip())
    return pd.DataFrame(data)

In [3]:
root = "../data/opus_opensub/en-pl.txt/"
files = ('OpenSubtitles.en-pl.en', '/OpenSubtitles.en-pl.pl')
df = upload_to_dataframe(root, files, 600000)

### 2. EDA & Sanitazation

#### 2.1 English Non-Ascii Sentences

In [4]:
def nonascii_list(df_series, is_pol):
    if is_pol:
        pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
        requir = lambda x: x.isascii() or x in pl_chars
    else:
        requir = lambda x: x.isascii()
        
    text_all = ''.join(df_series)
    unique_dct = {x for x in text_all if not requir(x)}
    return sorted(list(unique_dct))

In [5]:
print(nonascii_list(df['eng_text'], False))

['\x80', '\x81', '\x82', '\x83', '\x84', '\x85', '\x88', '\x8b', '\x8c', '\x8e', '\x8f', '\x91', '\x94', '\x96', '\x98', '\x99', '\x9c', '\x9d', '\xa0', '¡', '¢', '£', '¤', '¥', '¦', '§', '¨', '©', 'ª', '¬', '\xad', '°', '±', '²', '³', '´', 'µ', '¶', '·', '¸', '¹', 'º', '¼', '½', '¾', '¿', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Ç', 'É', 'Ê', 'Ì', 'Ï', 'Ð', 'Ñ', 'Ô', 'Ö', 'Ø', 'Ü', 'Ý', 'Þ', 'ß', 'à', 'á', 'â', 'ã', 'ä', 'å', 'ç', 'è', 'é', 'ê', 'ì', 'í', 'î', 'ï', 'ð', 'ñ', 'ó', 'ô', 'ö', 'ù', 'ú', 'û', 'ü', 'ý', 'þ', 'Ă', 'ď', 'ę', 'Ł', 'ń', 'ő', 'œ', 'ś', 'Š', 'š', 'Ј', 'С', 'й', '،', 'أ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ح', 'ر', 'س', 'ش', 'ع', 'غ', 'ف', 'ك', 'ل', 'م', 'ن', 'و', 'ي', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '\u200b', '‒', '–', '—', '‚', '€', '™', '─', '☻', '♥', '♪', '慹', '拢', '檛', '鈥', 'ﬁ', 'ﬂ']


In [6]:
is_ascii = lambda text: text.isascii()
maska = df['eng_text'].apply(is_ascii)
df_non_ascii = df[~maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.head()

(4595, 2)


,eng_text,pol_text
0,¿You had never seen A crab suedes? Them there ...,- Nigdy wcześniej nie widziałaś raków?
1,"He hears ¿, where you live?",Nigdy wcześniej nie byłam w lesie.
2,"Boys ¿, they are there?",Jesteście tam chłopaki?
3,- Te atrapé. - No es gracioso. Your he catches...,- Ale twoje mina była.
4,"Your also ¿, it is not like that?","- Tak, niezła jest"


In [7]:
df_2 = df[maska].reset_index(drop=True)
df_2.shape

(595405, 2)

#### 2.2 Polish Non-Ascii Sentences

In [8]:
pl_chars = set("ąęćłńóśźżĄĘĆŁŃÓŚŹŻ")
requir = lambda let: not let.isascii() and let not in pl_chars
maska = df_2['pol_text'].apply(lambda snt: any([requir(let) for let in snt]))
df_non_ascii = df_2[maska].copy().reset_index(drop=True)
print(df_non_ascii.shape)
df_non_ascii.sample(10)

(1227, 2)


,eng_text,pol_text
701,Your friend Mr Lincoln had his Taylors and Pai...,"Twój przyjaciel, pan Lincoln, tež mial swoich ..."
453,Big as life. Sparkling away under the old sun.,"Naturalnej wielkošci, polyskujaca w sloncu."
287,- It's nothing at all.,– Niezupełnie.
944,My Fuhrer!,Meine führer!
1220,Hotel de Paris.,W Hôtel de Paris.
525,Finally they think it's quite a bill.,"Wreszcie uznaja, že projekt jest šwietny."
410,"Some wealthy, influential citizen merely to cu...","Jakiemuš wplywowemu bogaczowi, by kupic jego l..."
550,Much better.,Dužo lepiej.
91,So you still believe that you are sent by the ...,"Ci±gle wierzysz, jakoby posyłał cię Bóg?"
1088,Glad to see the skies of blue Have turned into...,"¶ Cieszę się, gdy widzę jak błekitne niebo ¶ Z..."


In [9]:
df_3 = df_2[~maska].reset_index(drop=True)
print(df_3.shape)
df_3.sample(5)

(594178, 2)


,eng_text,pol_text
271116,We're gonna give you $500.,Damy ci 500 dolarów.
79266,These are all friends.,Tu wszyscy są przyjaciółmi.
41221,What?,Co?
415021,"Come, Mr. Cameron. Take this man away!",Niech pan zabierze tego człowieka!
379609,"Oh, yes, London.","Ach tak, Londyn."


In [10]:
print(nonascii_list(df_3['eng_text'], False))
print(nonascii_list(df_3['pol_text'], True))

[]
[]


In [11]:
df_3.iloc[74182]['eng_text']

'The next time I...'

In [12]:
df_3.iloc[74182]['pol_text']

'Następnym razem...'

#### 2.3 Short Or Empty Examples

In [13]:
mask1 = df_3['eng_text'].str.len() >= 3
mask2 = df_3['pol_text'].str.len() >= 3

In [14]:
df_4 = df_3[mask1 & mask2].reset_index(drop=True)
df_4.head()

,eng_text,pol_text
0,"Previously on ""The Blacklist""...",/W poprzednich odcinkach: /
1,- You want to call your daddy?,- Chcesz zadzwonić do taty?
2,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
3,Okay.,Dobrze.
4,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża."


In [15]:
df_4.shape

(593894, 2)

#### 2.4 Dialog like sentences [starts with - ]

In [16]:
is_dialog = lambda snt: bool(re.match(r"^ *- *", snt))
mask = df_4['eng_text'].apply(is_dialog) | df_4['pol_text'].apply(is_dialog)
df_dialogs = df_4[mask].copy().reset_index(drop=True)
df_dialogs.head()

,eng_text,pol_text
0,- You want to call your daddy?,- Chcesz zadzwonić do taty?
1,"- Yeah, I want to tell him I'm okay.","- Tak, powiem, że wszystko w porządku."
2,- 45 seconds.,Szybciej! 45 sekund.
3,It's okay. It's okay. It's okay.,- Już dobrze.
4,It's okay. You're okay. You're okay.,- Odpoczywaj.


In [17]:
def fix_dialog(snt):
    return re.sub(r"^ *- *", "", snt)

df_4['eng_text'] = df_4['eng_text'].apply(fix_dialog).reset_index(drop=True)
df_4['pol_text'] = df_4['pol_text'].apply(fix_dialog).reset_index(drop=True)
df_4.sample(5)

,eng_text,pol_text
543540,You must have faith in him.,Musisz w nim pokładać wiarę.
336391,I do.,Tak.
549006,I'll pay Ivey back for this if it's the last t...,Odpłacę Ivey'emu.
226664,Why not?,Czemu nie.
74830,"If I' m not mistaken, that's blood.","Jeśli się nie mylę, to jest krew."


#### 2.5 Broken Dialog like sentences [somehwere has -]

# Do naprawy (lepszy re przed tokenziacja)

In [18]:
broken_dialog = lambda snt: bool(re.match(r".* +- +.*", snt))
mask1 = df_4['eng_text'].apply(broken_dialog)
mask2 = df_4['pol_text'].apply(broken_dialog)

In [19]:
df_4[mask1 | mask2].sample(5)

,eng_text,pol_text
433552,A special brand? - Yes.,Wykonane na zamówienie?
285113,"Well, I did catch a glimpse of her some time ago.",Ujrzałem ją przez moment jakiś czas temu. - I...
439588,I wonder if we could tell him that... - That i...,"Mogę powiedzieć że nie powiodło się, i nie dzi..."
503723,And I've got to be this way just once to fix i...,Popełniłam w życiu wielki błąd i muszę to zrob...
113700,You're smart enough to hide that note.,"Czego chcesz? - Jesteś sprytna, ukryłaś banknot."


In [20]:
df_5 = df_4[~mask1 & ~mask2].reset_index(drop=True)

In [21]:
broken_dialog = lambda snt: bool(re.match(r".*-.*", snt))

In [22]:
df_5.sample(5)

,eng_text,pol_text
552966,Willie Garzah.,Willie Garzah.
513234,"Oh, maybe.",Może.
175889,It wouldn't be considered good manners to take...,"Uznano by za niestosowne pragnąć kobiety, któr..."
105915,I was told beware my wedding night.,Ostrzegano mnie przed dniem mego ślubu..
322414,He told me Whitey was dead before he took me h...,"Powiedział mi, że Whitey nie żyje jeszcze zani..."


#### 2.6 Different Last Case

In [23]:
mask = df_5['eng_text'].str[-1] != df_5['pol_text'].str[-1]
mask.sum()

57558

In [24]:
df_6 = df_5[~mask].reset_index(drop=True)

#### 2.7 Text length difference

In [25]:
mask = df_6['eng_text'].str.split
df_6['len_ratio'] = df_6['eng_text'].str.split().str.len() / df_6['pol_text'].str.split().str.len()
df_6.head()

,eng_text,pol_text,len_ratio
0,You want to call your daddy?,Chcesz zadzwonić do taty?,1.500000
1,"Yeah, I want to tell him I'm okay.","Tak, powiem, że wszystko w porządku.",1.333333
2,Okay.,Dobrze.,1.000000
3,Lizzy... Be careful of your husband.,"Lizzy, uważaj na męża.",1.500000
4,I can only lead you to the truth.,/Mogę naprowadzić cię na prawdę.,1.600000


In [26]:
thres_left = df_6["len_ratio"].quantile(0.05)
thres_right = df_6["len_ratio"].quantile(0.95)
thres_left, thres_right

(0.75, 2.5)

In [27]:
mask = (df_6["len_ratio"] >= thres_left) & (df_6["len_ratio"] <= thres_right)
mask.sum()

472209

In [28]:
df_7 = df_6[mask].reset_index(drop=True)
df_7.sample(5)

,eng_text,pol_text,len_ratio
120221,"If you're not shot by the time you're 50, you ...","Jeśli nie zginiesz przed piećdziesiatką, skońc...",1.666667
21732,I shouldn't like that.,Nie powinnam była tego mówić.,0.800000
408066,"Oh, sure.",Jasne.,2.000000
304036,Just level off and head straight for home.,Wyrównaj poziom i leć prosto do domu.,1.142857
349903,"It's been such fun, Jim. Yes.","To była świetna zabawa, Jim.",1.200000


In [29]:
df_7.sample(5)

,eng_text,pol_text,len_ratio
252661,Roy?,Roy?,1.0
330928,At least I think so.,"Ja jestem pewien, że tak.",1.0
147819,He doesn't look like a criminal.,Nie wygląda na przestępcę.,1.5
464938,Any kind...,Obojętnie...,2.0
235062,"Oh, uh...hello.",Cześć.,2.0


#### 2.8 Sentences starting with * fix

In [30]:
fix_star = lambda snt: re.sub(r"^ *[*].*", "", snt)
df_7['eng_text'] = df_7['eng_text'].apply(fix_star)
df_7['pol_text'] = df_7['pol_text'].apply(fix_star)

#### 2.9 Action Comments

In [31]:
def mask_distinct(df, col1, col2, chars):
    chars_escaped = "".join(re.escape(c) for c in chars)
    templ = f"[{chars_escaped}]"

    m1 = df[col1].str.contains(templ, regex=True, na=False)
    m2 = df[col2].str.contains(templ, regex=True, na=False)

    return m1 ^ m2

chars = '[]()/\*+#$"️'
mask = mask_distinct(df_7, "eng_text", "pol_text", chars)

df_8 = df_7[~mask].reset_index(drop=True)
df_8.shape

(467550, 3)

In [32]:
df_8.sample(5)

,eng_text,pol_text,len_ratio
321923,"Doctor, Teddy, signature...","Lekarz, Teddy, podpis...",1.000000
267632,"Who, from this hour forth, can ask anything in...","Który od tej pory może prosić o wszystko, co w...",1.083333
343646,What about your passengers?,A co z pasażerami?,1.000000
317472,She is difficult.,Jest trudna.,1.500000
258187,He's getting more like his father every day.,Coraz bardziej przypomina ojca.,2.000000


#### 2.10  " ` " ---> " ' " & del " ~ "

In [33]:
wanted_case = '`'
mask = df_8["eng_text"].apply(lambda x: wanted_case in x)
df_8[mask]

,eng_text,pol_text,len_ratio
214145,"Revolution had broken out, her diplomats sued ...","Rewolucja na tyłach frontu, dąży do zawarcia p...",1.277778
214152,We`ll examine it.,Zbadajmy go.,1.500000
214157,What do you think you`re doing?,Wstawaj! Co ty robisz?,1.500000
214161,Where`s your hand grenade?,Gdzie masz swój granat?,1.000000
214163,Let `em have it!,Niech je poczują!,1.333333
...,...,...,...
214943,We don`t want to hate one another.,Bez nienawiści ani pogardy.,1.750000
214944,There`s room for everyone.,Każdy ma swe miejsce.,1.000000
214947,"Greed has poisoned men`s souls, has barricaded...","Chciwość zatruła nasze dusze, wzniosła mury ni...",1.333333
214963,You don`t hate.,Przestańcie nienawidzieć.,1.500000


In [34]:
df_8["eng_text"] = df_8["eng_text"].str.replace("`", "'", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("`", "'", regex=False)

In [35]:
df_8["eng_text"] = df_8["eng_text"].str.replace("~", "", regex=False)
df_8["pol_text"] = df_8["pol_text"].str.replace("~", "", regex=False)

#### 2.11 Del rows with specific case

In [36]:
def del_row_with_char(df, eng_col, pol_col, chars):
    df_proc = df.copy()
    for char in chars:
        mask1 = df_proc[eng_col].apply(lambda x: char in x)
        mask2 = df_proc[pol_col].apply(lambda x: char in x)
        df_proc = df_proc[~(mask1 | mask2)].reset_index(drop=True)
    return df_proc
    

chars = "=+*@#;|_}{"
df_9 = del_row_with_char(df_8, "eng_text", "pol_text", chars)

#### 2.12 Del rows with dialog -

In [37]:
mask1 = df_9["eng_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
mask2 = df_9["pol_text"].apply(lambda x: bool(re.search(r' +-[^ ]+', x)))
df_data = df_9[~(mask1 | mask2)].reset_index(drop=True)

#### 3. Data Tokenization

In [38]:
uniq_cases = set("".join(df_data["eng_text"].astype(str)))
tab1 = sorted(list(uniq_cases), reverse=True)
print(tab1)

['z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [39]:
uniq_cases = set("".join(df_data["pol_text"].astype(str)))
tab2 = sorted(list(uniq_cases), reverse=True)
print(tab2)

['ż', 'Ż', 'ź', 'Ź', 'ś', 'Ś', 'ń', 'Ń', 'ł', 'Ł', 'ę', 'Ę', 'ć', 'Ć', 'ą', 'Ą', 'ó', 'Ó', 'z', 'y', 'x', 'w', 'v', 'u', 't', 's', 'r', 'q', 'p', 'o', 'n', 'm', 'l', 'k', 'j', 'i', 'h', 'g', 'f', 'e', 'd', 'c', 'b', 'a', ']', '[', 'Z', 'Y', 'X', 'W', 'V', 'U', 'T', 'S', 'R', 'Q', 'P', 'O', 'N', 'M', 'L', 'K', 'J', 'I', 'H', 'G', 'F', 'E', 'D', 'C', 'B', 'A', '?', ':', '9', '8', '7', '6', '5', '4', '3', '2', '1', '0', '/', '.', '-', ',', ')', '(', "'", '%', '$', '"', '!', ' ']


In [40]:
def tokenize_eng(snt):
    snt = snt.lower()
    snt = re.sub("(?<! )'(?! )", r" '", snt)
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [41]:
def tokenize_pol(snt):
    snt = snt.lower()
    snt = re.sub(r"""([\]?:)\[(/\.\-,%\$"!])""", r" \1 ", snt)
    snt = re.sub(r"\s+", " ", snt).strip()
    return re.split(r'\s+', snt.strip())

In [42]:
df_data = df_data.drop(['len_ratio'], axis=1)

#### 3.1 Splitting and trimming on quantile

In [43]:
df_data['eng_split'] = df_data['eng_text'].apply(tokenize_eng)
df_data['pol_split'] = df_data['pol_text'].apply(tokenize_pol)
df_data.sample(10)

,eng_text,pol_text,eng_split,pol_split
207261,"The hot-tempered young fool must carry a gun, ...","Ten młody wariat chce nosić broń, bo w okolicy...","[the, hot, -, tempered, young, fool, must, car...","[ten, młody, wariat, chce, nosić, broń, ,, bo,..."
4263,"""Bare your body, Brother!""","""Obnaż swoje ciało, bracie!""","["", bare, your, body, ,, brother, !, ""]","["", obnaż, swoje, ciało, ,, bracie, !, ""]"
432673,Mr. Gagin!,Panie Gagin!,"[mr, ., gagin, !]","[panie, gagin, !]"
34104,"Say, you want to draw another paycheck, don't ...","Chcesz dostać kolejną wypłatę, prawda?","[say, ,, you, want, to, draw, another, paychec...","[chcesz, dostać, kolejną, wypłatę, ,, prawda, ?]"
45450,"A, I got A.",Dostałem piątkę.,"[a, ,, i, got, a, .]","[dostałem, piątkę, .]"
335069,Really?,Naprawdę?,"[really, ?]","[naprawdę, ?]"
399162,Do I know you?,Znam pana?,"[do, i, know, you, ?]","[znam, pana, ?]"
267746,Bambi!,Bambi!,"[bambi, !]","[bambi, !]"
60068,As my own child.,Jak na swoje własne dziecko.,"[as, my, own, child, .]","[jak, na, swoje, własne, dziecko, .]"
209403,I've got it.,Są. Będą na pewno.,"[i, 've, got, it, .]","[są, ., będą, na, pewno, .]"


In [44]:
thres1 = df_data['eng_split'].str.len().quantile(0.99)
mask1 = df_data['eng_split'].str.len() > thres1

thres2 = df_data['pol_split'].str.len().quantile(0.99)
mask2 = df_data['pol_split'].str.len() > thres2
print(mask1.sum(), mask2.sum(), (mask1 | mask2).sum())

df_data = df_data[~(mask1 | mask2)].reset_index(drop=True)

4113 3960 5510


#### 3.2 BPE Init. Uniq_freq & Rozklad chars

In [45]:
uniq_freq_eng = Counter(df_data['eng_split'].explode())
uniq_freq_pol = Counter(df_data['pol_split'].explode())

In [46]:
len(uniq_freq_eng), len(uniq_freq_pol)

(40087, 113603)

In [455]:
char_factor = lambda word: [x for x in word] + ['</w>']
vocab_chars_eng = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
vocab_chars_pol = [[char_factor(k), v] for k, v in uniq_freq_pol.most_common()]
print(len(vocab_chars_eng), vocab_chars_eng[:5], '\n')
print(len(vocab_chars_pol), vocab_chars_pol[:5]) 

40087 [[['.', '</w>'], 435802], [[',', '</w>'], 177291], [['y', 'o', 'u', '</w>'], 134548], [['i', '</w>'], 122767], [['t', 'h', 'e', '</w>'], 89859]] 

113603 [[['.', '</w>'], 413780], [[',', '</w>'], 183948], [['?', '</w>'], 83444], [['n', 'i', 'e', '</w>'], 82694], [['t', 'o', '</w>'], 53190]]


#### 3.3 BPE Tokenizer most common pairs

In [450]:
# class BPETokenizer():
#     def __init__(self, vocab_chrs, vocab_size, is_tgt):
#         self.vocab_chrs = vocab_chrs
#         self.vocab_size = vocab_size
#         self.vocab = self.vocab_init(vocab_chrs, is_tgt)
#         self.base_size = len(self.vocab)

#     def vocab_init(self, vocab_chrs, is_tgt):
#         vocab = {'<pad>': 0, '<unk>': 1, '<eos>': 2, '</w>': 3}
#         if is_tgt:
#             vocab['<bos>'] = 4
            
#         uniq_toks = {tok for toks, _ in vocab_chrs for tok in toks}
#         uniq_toks.remove('</w>')
#         for i, tok in enumerate(uniq_toks, len(vocab)):
#             vocab[tok] = i
#         return vocab                

#     def train_bpe(self):
#         for i in range(self.base_size, self.vocab_size):
#             pair = self.most_common_pair()
#             self.vocab[pair] = i
#             pair_con = re.sub('\s+', '', pair)
#             self.replace_pairs(pair, pair_con)

#     def most_common_pair(self):
#         pair_counts = Counter()
#         for toks, freq in self.vocab_chrs:
#             for pair in map(' '.join, zip(toks, toks[1:])):
#                 pair_counts[pair] += freq
#         return pair_counts.most_common(1)[0][0]

#     def replace_pairs(self, pair, pair_con):
#         for i, (toks, _) in enumerate(self.vocab_chrs):
#             self.vocab_chrs[i][0] = ' '.join(toks).replace(pair, pair_con).split()    

In [454]:
class BPETokenizer():
    def __init__(self, vocab_chrs, vocab_size, is_tgt):
        self.vocab_chrs = vocab_chrs
        self.vocab_size = vocab_size
        self.vocab = self.vocab_init(vocab_chrs, is_tgt)
        self.base_size = len(self.vocab)

    def vocab_init(self, vocab_chrs, is_tgt):
        vocab = {'<pad>': 0, '<unk>': 1, '<eos>': 2, '</w>': 3}
        if is_tgt:
            vocab['<bos>'] = 4
            
        uniq_toks = {tok for toks, _ in vocab_chrs for tok in toks}
        uniq_toks.remove('</w>')
        for i, tok in enumerate(uniq_toks, len(vocab)):
            vocab[tok] = i
        return vocab                

    def train_bpe(self):
        for i in range(self.base_size, self.vocab_size):
            pair = self.most_common_pair()
            pair_spc, pair_con = ' '.join(pair), ''.join(pair)
            self.vocab[pair_spc] = i
            self.replace_pairs(pair, pair_spc, pair_con)

    def most_common_pair(self):
        pair_counts = defaultdict(int)
        for toks, freq in self.vocab_chrs:
            for pair in zip(toks, toks[1:]):
                pair_counts[pair] += freq
        return max(pair_counts, key=pair_counts.get)

    def replace_pairs(self, pair, pair_spc, pair_con):
        for i, (toks, _) in enumerate(self.vocab_chrs):
            if pair in zip(toks, toks[1:]):
                self.vocab_chrs[i][0] = ' '.join(toks).replace(pair_spc, pair_con).split()

In [456]:
tokenizer_eng = BPETokenizer(vocab_chars_eng, 8000, False)
t = time.time()
tokenizer_eng.train_bpe()
print(f"{time.time()-t:.2f} seconds")

498.02 seconds


In [461]:
tab = ['a', 'b', 'r', 'a', 'c', 'a', 'd', 'a', 'br', 'a', '</w>']
print(list(zip(tab, tab[1:])))
if ('a', 'b') in zip(tab, tab[1:]):
    print(' '.join(tab).replace('a b', 'ab').split())

[('a', 'b'), ('b', 'r'), ('r', 'a'), ('a', 'c'), ('c', 'a'), ('a', 'd'), ('d', 'a'), ('a', 'br'), ('br', 'a'), ('a', '</w>')]
['ab', 'r', 'a', 'c', 'a', 'd', 'abr', 'a', '</w>']


In [471]:
tab2 = list(zip(tab, tab[1:]))
print(tab2)

[('a', 'b'), ('b', 'r'), ('r', 'a'), ('a', 'c'), ('c', 'a'), ('a', 'd'), ('d', 'a'), ('a', 'br'), ('br', 'a'), ('a', '</w>')]


## edytuj liste z tuplami i trzymaj tupla w slowniku

In [472]:
tab2.index(('a', 'b'))

0

In [458]:
[[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]

[[['.', '</w>'], 435802],
 [[',', '</w>'], 177291],
 [['y', 'o', 'u', '</w>'], 134548],
 [['i', '</w>'], 122767],
 [['t', 'h', 'e', '</w>'], 89859],
 [['?', '</w>'], 84640],
 [['t', 'o', '</w>'], 73192],
 [['a', '</w>'], 61661],
 [["'", 's', '</w>'], 57045],
 [['i', 't', '</w>'], 55074],
 [["'", 't', '</w>'], 50411],
 [['t', 'h', 'a', 't', '</w>'], 43020],
 [['a', 'n', 'd', '</w>'], 41124],
 [['o', 'f', '</w>'], 39514],
 [['i', 'n', '</w>'], 32142],
 [['!', '</w>'], 30794],
 [['m', 'e', '</w>'], 29977],
 [['w', 'h', 'a', 't', '</w>'], 26868],
 [['i', 's', '</w>'], 26378],
 [['h', 'e', '</w>'], 26149],
 [['f', 'o', 'r', '</w>'], 23935],
 [['w', 'e', '</w>'], 23370],
 [['t', 'h', 'i', 's', '</w>'], 20366],
 [['y', 'o', 'u', 'r', '</w>'], 20158],
 [['b', 'e', '</w>'], 20137],
 [['m', 'y', '</w>'], 20105],
 [["'", 'l', 'l', '</w>'], 20011],
 [['h', 'a', 'v', 'e', '</w>'], 19830],
 [['d', 'o', 'n', '</w>'], 18888],
 [['d', 'o', '</w>'], 18663],
 [['o', 'n', '</w>'], 18052],
 [['n', 'o', '</

In [457]:
tokenizer_eng.vocab

{'<pad>': 0,
 '<unk>': 1,
 '<eos>': 2,
 '</w>': 3,
 '9': 4,
 'h': 5,
 '4': 6,
 'z': 7,
 '.': 8,
 't': 9,
 '%': 10,
 '1': 11,
 'u': 12,
 'w': 13,
 '7': 14,
 '[': 15,
 '3': 16,
 'c': 17,
 '5': 18,
 'd': 19,
 'i': 20,
 '8': 21,
 '$': 22,
 ']': 23,
 ')': 24,
 'v': 25,
 'l': 26,
 'k': 27,
 ',': 28,
 'm': 29,
 'r': 30,
 'j': 31,
 'q': 32,
 's': 33,
 '"': 34,
 '6': 35,
 'x': 36,
 '?': 37,
 'e': 38,
 'f': 39,
 'y': 40,
 '/': 41,
 '(': 42,
 '-': 43,
 ':': 44,
 'o': 45,
 '2': 46,
 "'": 47,
 '0': 48,
 'g': 49,
 'n': 50,
 'p': 51,
 '!': 52,
 'a': 53,
 'b': 54,
 'e </w>': 55,
 '. </w>': 56,
 't </w>': 57,
 't h': 58,
 's </w>': 59,
 'o u': 60,
 'n </w>': 61,
 'd </w>': 62,
 ', </w>': 63,
 'e r': 64,
 'y </w>': 65,
 'y ou': 66,
 'o </w>': 67,
 'i n': 68,
 'you </w>': 69,
 'i </w>': 70,
 'a n': 71,
 'l </w>': 72,
 'r </w>': 73,
 'th e</w>': 74,
 'g </w>': 75,
 'a t</w>': 76,
 '? </w>': 77,
 'a </w>': 78,
 'l l</w>': 79,
 't o</w>': 80,
 'in g</w>': 81,
 'er </w>': 82,
 'a r': 83,
 'f </w>': 84,
 'i t

#### 3.4 BPE Encoder propably should add to the BPE Class Later

In [107]:
test_word = ' '.join(char_factor("running"))
test_word

'r u n n i n g </w>'

In [175]:
# pair = 'g </w>'
pair = '['
sub_pair = lambda x: re.sub('\s+', '', x)
re.sub(rf"( |^){re.escape(pair)}( |$)", rf"\g<1>{sub_pair(pair)}\g<2>", "r u n n i n g </w>")

'r u n n i n g </w>'

In [196]:
def word_encoder(word, vocab):
    word_str = ' '.join(char_factor(word)) #do zmiany uzywanie char_factor spoza funkcji
    sub_key = lambda x: re.sub('\s+', '', x)
    
    for k, v in vocab.items():
        if bool(re.search(rf"( |^){re.escape(k)}( |$)", word_str)):
            print(k, v)
            print(word_str)
            word_str = re.sub(rf"( |^){re.escape(k)}( |$)", rf"\g<1>{sub_key(k)}\g<2>", word_str)
            print(word_str, '\n')
    return word_str

In [218]:
tokenizer_eng.vocab

{'<pad>': 0,
 '<unk>': 1,
 '<eos>': 2,
 '</w>': 3,
 '9': 4,
 'h': 5,
 '4': 6,
 'z': 7,
 '.': 8,
 't': 9,
 '%': 10,
 '1': 11,
 'u': 12,
 'w': 13,
 '7': 14,
 '[': 15,
 '3': 16,
 'c': 17,
 '5': 18,
 'd': 19,
 'i': 20,
 '8': 21,
 '$': 22,
 ']': 23,
 ')': 24,
 'v': 25,
 'l': 26,
 'k': 27,
 ',': 28,
 'm': 29,
 'r': 30,
 'j': 31,
 'q': 32,
 's': 33,
 '"': 34,
 '6': 35,
 'x': 36,
 '?': 37,
 'e': 38,
 'f': 39,
 'y': 40,
 '/': 41,
 '(': 42,
 '-': 43,
 ':': 44,
 'o': 45,
 '2': 46,
 "'": 47,
 '0': 48,
 'g': 49,
 'n': 50,
 'p': 51,
 '!': 52,
 'a': 53,
 'b': 54,
 'e </w>': 55,
 '. </w>': 56,
 't </w>': 57,
 't h': 58,
 's </w>': 59,
 'o u': 60,
 'n </w>': 61,
 'd </w>': 62,
 ', </w>': 63,
 'e r': 64,
 'y </w>': 65,
 'y ou': 66,
 'o </w>': 67,
 'i n': 68,
 'you </w>': 69,
 'i </w>': 70,
 'a n': 71,
 'l </w>': 72,
 'r </w>': 73,
 'th e</w>': 74,
 'g </w>': 75,
 'a t</w>': 76,
 '? </w>': 77,
 'a </w>': 78,
 'l l</w>': 79,
 't o</w>': 80,
 'in g</w>': 81,
 'a r': 82,
 'f </w>': 83,
 'i t</w>': 84,
 'm e

In [333]:
def encode_word(word, vocab):
    word_str = ' '.join(word)
    sub_key = lambda x: re.sub('\s+', '', x)
    for key in vocab.keys():
        pat = ' ' + key + ' '
        if pat in word_str:
            print(key)
            print(word_str, '\n')
            word_str = word_str.replace(key, sub_key(key))
            # print(key)
            # print(word_str, '\n')
    return word_str

In [336]:
encode_word(char_factor(' running '), tokenizer_eng.vocab)

n
  r u n n i n g   </w> 

g
  r u n n i n g   </w> 

r
  r u n n i n g   </w> 

u
  r u n n i n g   </w> 

i
  r u n n i n g   </w> 

i n
  r u n n i n g   </w> 

u n
  r u n n in g   </w> 



'  r un n in g   </w>'

In [269]:
test_snt = "Yesterday I visited zoo with my family, it was super fun!"
test_snt_split = tokenize_eng(test_snt)
print(test_snt_split)

['yesterday', 'i', 'visited', 'zoo', 'with', 'my', 'family', ',', 'it', 'was', 'super', 'fun', '!']


In [270]:
[[x for x in word] + ['</w>'] for word in test_snt_split]

[['y', 'e', 's', 't', 'e', 'r', 'd', 'a', 'y', '</w>'],
 ['i', '</w>'],
 ['v', 'i', 's', 'i', 't', 'e', 'd', '</w>'],
 ['z', 'o', 'o', '</w>'],
 ['w', 'i', 't', 'h', '</w>'],
 ['m', 'y', '</w>'],
 ['f', 'a', 'm', 'i', 'l', 'y', '</w>'],
 [',', '</w>'],
 ['i', 't', '</w>'],
 ['w', 'a', 's', '</w>'],
 ['s', 'u', 'p', 'e', 'r', '</w>'],
 ['f', 'u', 'n', '</w>'],
 ['!', '</w>']]

In [135]:
def connect_pairs(vocab_chars, pair):
    vocab_proc = vocab_chars.copy()
    pair_con = re.sub('\s+', '', pair)
    for i, (toks, _) in enumerate(vocab_proc):
        toks = " ".join(toks).replace(pair, pair_con).split()
        vocab_proc[i][0] = toks
    return vocab_proc

In [221]:
def most_common_pair(vocab_chars):
    pair_counts = Counter()
    for toks, freq in vocab_chars:
        for pair in map(' '.join, zip(toks, toks[1:])):
            pair_counts[pair] += freq
    return pair_counts.most_common(1)[0][0]

In [226]:
vocab_chars_eng

[[['.', '</w>'], 435802],
 [[',', '</w>'], 177291],
 [['y', 'o', 'u', '</w>'], 134548],
 [['i', '</w>'], 122767],
 [['t', 'h', 'e', '</w>'], 89859],
 [['?', '</w>'], 84640],
 [['t', 'o', '</w>'], 73192],
 [['a', '</w>'], 61661],
 [["'", 's', '</w>'], 57045],
 [['i', 't', '</w>'], 55074],
 [["'", 't', '</w>'], 50411],
 [['t', 'h', 'a', 't', '</w>'], 43020],
 [['a', 'n', 'd', '</w>'], 41124],
 [['o', 'f', '</w>'], 39514],
 [['i', 'n', '</w>'], 32142],
 [['!', '</w>'], 30794],
 [['m', 'e', '</w>'], 29977],
 [['w', 'h', 'a', 't', '</w>'], 26868],
 [['i', 's', '</w>'], 26378],
 [['h', 'e', '</w>'], 26149],
 [['f', 'o', 'r', '</w>'], 23935],
 [['w', 'e', '</w>'], 23370],
 [['t', 'h', 'i', 's', '</w>'], 20366],
 [['y', 'o', 'u', 'r', '</w>'], 20158],
 [['b', 'e', '</w>'], 20137],
 [['m', 'y', '</w>'], 20105],
 [["'", 'l', 'l', '</w>'], 20011],
 [['h', 'a', 'v', 'e', '</w>'], 19830],
 [['d', 'o', 'n', '</w>'], 18888],
 [['d', 'o', '</w>'], 18663],
 [['o', 'n', '</w>'], 18052],
 [['n', 'o', '</

In [102]:
def most_common_pair(vocab_chars):
    freq_pair = defaultdict(int)
    for toks, freq in vocab_chars:
        toks_pairs = [" ".join(pair) for pair in zip(toks, toks[1:])]
        for pair in toks_pairs:
            freq_pair[pair] += freq
    return max(freq_pair.items(), key=lambda x: x[1])[0]

In [190]:
pair_counts = Counter()
pair_counts[('t','h')] += 10
pair_counts[('t','h')] += 10
pair_counts.most_common(1)

[(('t', 'h'), 20)]

In [394]:
vocab_chars_eng2 = [[char_factor(k), v] for k, v in uniq_freq_eng.most_common()]
tokenizer_eng_test = BPETokenizer(vocab_chars_eng2, 100, False)
t = time.time()
tokenizer_eng_test.train_bpe()
print(f"{time.time()-t:.2f} seconds")

4.49 seconds
